<a href="https://colab.research.google.com/github/claregelbrugge/rush-sales-data/blob/main/RUSH_Case_Study_GB885.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RUSH Sales Data Analysis

## Business objective:

You work as an analyst for RUSH, a globally renowned sportswear and footwear brand known for its innovative designs and performance-oriented products. The company stores its raw sales data as a collection of three tables:

*   TABLE_PRODUCTS
*   TABLE_RETAILERS
*   TABLE_SALES



This data contains the number of units sold, the total sales revenue, the location of the sales, type of product sold as well as other relevant information.

The VP of US Sales has tasked you with analyzing sales data for trends and insights that will help company leadership understand the market and opportunities for growth.

They have also asked you to answer several business questions.

### Load the data

In [ ]:
# import pandas as pd
import pandas as pd

In [ ]:
# load the three tables
products=pd.read_csv('TABLE_PRODUCTS_885.csv')
retailers=pd.read_csv('TABLE_RETAILER_885.csv')
sales=pd.read_csv('TABLE_SALES_885.csv')

In [ ]:
# check that the products table loaded correctly
products.head(4)

In [ ]:
# separate the columns
products = pd.read_csv("TABLE_PRODUCTS_885.csv", sep="|")

In [ ]:
# check that the columns separate products correctly
products.head()

### Inspect tables

In [ ]:
# check the retailer table loaded correctly
retailers.head()

In [ ]:
# check that the sales table loaded correctly
sales.head()

In [ ]:
# understand the data shape
products.shape

In [ ]:
# understand the data shape of retailers table
retailers.shape

In [ ]:
# understand the data shape of sales table
sales.shape

### Merge tables

In [ ]:
# merge the data tables on product_id
sales_data = sales.merge(products, on='PRODUCT_ID', how='left')

In [ ]:
# merge the data with retailer on the retailer_id
sales_data = sales_data.merge(retailers, on='RETAILER_ID', how='left')

In [ ]:
# check all the data together
sales_data.head(10)

In [ ]:
# check that the merge worked
sales_data.shape

# EDA/Data Preparation:


###Check for nulls

In [ ]:
# check for nulls
sales_data.isna().sum()

In [ ]:
# find the rows with missing prices
sales_data[sales_data['PRICE_PER_UNIT'].isna()]

In [ ]:
# find the median for this specific product's price
sales_data.loc[sales_data['PRODUCT_ID'] == 20, 'PRICE_PER_UNIT'].median()

In [ ]:
# replace missing prices with the median price for product_id=20
sales_data.loc[98, 'PRICE_PER_UNIT'] = 44
sales_data.loc[99, 'PRICE_PER_UNIT'] = 44

In [ ]:
# check that the change was made
sales_data.loc[[98,99]]

The two missing price values were replaced with the median price for the corresponding product, which was $44.

In [ ]:
# find the rows with missing retailer
sales_data[sales_data['RETAILER'].isna()]

This item is missing retailer, region, state, and city and it has a suspicious 999999 value for retailer_id.

In [ ]:
# change the missing values to "Unknown"
sales_data.loc[1534, ['RETAILER_ID','RETAILER', 'REGION', 'STATE', 'CITY']] = 'Unknown'

In [ ]:
# make sure it worked
sales_data.loc[1534]

The row contained a placeholder retailer ID (999999999) and missing retailer and location information. Because the correct values could not be determined, these fields were changed to "Unknown" rather than being guessed.

In [ ]:
# check that no more remaining NAs
sales_data.isna().sum()

###Check for duplicated rows

In [ ]:
# check for duplicated rows
sales_data.duplicated().sum()

There are no duplicated rows.

###Check that data types are correct

In [ ]:
# check data types
sales_data.dtypes

In [ ]:
# convert invoice date to an actual date instead of object data type
sales_data['INVOICE_DATE'] = pd.to_datetime(sales_data['INVOICE_DATE'])

In [ ]:
# convert units sold to a numeric data type instead of object data type
sales_data['UNITS_SOLD'] = pd.to_numeric(sales_data['UNITS_SOLD'], errors='coerce')

In [ ]:
# check the data types again
sales_data.dtypes

In [ ]:
# check for NAs again after data types are changed
sales_data.isna().sum()

It seems that 2 NAs have popped up after the data type was changed for the units_sold column.

###Check for extreme outliers

In [ ]:
# check with describe to see if there are any major outliers
sales_data[['PRICE_PER_UNIT', 'UNITS_SOLD', 'OPERATING_MARGIN']].describe()

A price_per_unit value value of 99,999 was identified as an extreme and likely erroneous value. We need to replace with the median price for the corresponding product ($44).

In [ ]:
# find the item with 99999 price_per_unit value
sales_data[sales_data['PRICE_PER_UNIT'] == 99999]

In [ ]:
# replace this price per unit with the median used before
sales_data.loc[423, 'PRICE_PER_UNIT'] = 44

In [ ]:
# see if it worked
sales_data.loc[423]

Looking at the description table, units_sold also has a minimum of 0, which means there are rows that are 0 or NA, which should be examined.

In [ ]:
# check the missing values from units sold
sales_data[sales_data['UNITS_SOLD'].isna()]

Just keep the rows because no way to determine how many units sold.

In [ ]:
# check describe again to see if everything looks good
sales_data.describe()

The max order_id is 9648, yet the count of order_ids is 10,271. This probably means that there are duplicated order_ids.

In [ ]:
# check unique order_id
sales_data['ORDER_ID'].nunique()


In [ ]:
# check duplicated order_ids
sales_data['ORDER_ID'].duplicated().sum()

There are 623 repeated order_id values, but there are no completely duplicated rows. Repeated order IDs are likely due to orders containing multiple products, so these records were retained.

### Check unique column values for spelling and errors

In [ ]:
# look at unique product name values
sales_data['PRODUCT_NAME'].unique()

In [ ]:
# look at unique retailer values
sales_data['RETAILER'].unique()

In [ ]:
# look at unique regions
sales_data['REGION'].unique()

In [ ]:
# look at unique states
sales_data['STATE'].unique()

In [ ]:
# look at unique sales methods
sales_data['SALES_METHOD'].unique()

Outlet is spelled wrong here as 'ootlet', this needs to be changed.

In [ ]:
# fix how outlet is spelled
sales_data['SALES_METHOD'] = sales_data['SALES_METHOD'].replace('Ootlet', 'Outlet')

# Analysis/Business Questions


Three of the business questions provided by the sales team require sales data, so we need to build a column for total sales calculated from price_per_unit and units_sold.

In [ ]:
# calculate total sales revenue for each record and create a sales column
sales_data['SALES'] = sales_data['PRICE_PER_UNIT'] * sales_data['UNITS_SOLD']

In [ ]:
# check the column calculation
sales_data[['PRICE_PER_UNIT', 'UNITS_SOLD', 'SALES']].head()

###Question 1: What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?

In [ ]:
# create new sales variable for 2021
sales_2021 = sales_data[sales_data['YEAR'] == 2021]

# group by product name and take sum sales from each product category
product_sales = sales_2021.groupby('PRODUCT_NAME')['SALES'].sum()

# get the product name with the most sales in descending order
product_sales.sort_values(ascending=False).head(1).round(2)

**Men's street footwear** had the highest sales, with approximately **$23.2 million** in sales.

###Question 2: What state had the highest sales (in dollars) of women's products in 2021? How much was it?

In [ ]:
# check how product names are labelled for each sex
sales_data['PRODUCT_NAME'].unique()

In [ ]:
# create variable for women's category in 2021
women_2021 = sales_data[
    (sales_data['YEAR'] == 2021) &
    (sales_data['PRODUCT_NAME'].str.contains("Women's", na=False))
]

# group the 2021 women's sales data together
women_state_sales = women_2021.groupby('STATE')['SALES'].sum()

# see the highest sales value
women_state_sales.sort_values(ascending=False).head(1)

**Maine** was the state with the highest women's product sales of **2.1 million**.

###Question 3: What state had the highest sales (in dollars) of men's products in 2021? How much was it?

In [ ]:
# make a variable for men's category in 2021
men_2021 = sales_data[
    (sales_data['YEAR'] == 2021) &
    (sales_data['PRODUCT_NAME'].str.contains("Men's", na=False))
]

# group the 2021 men's sales data together
men_state_sales = men_2021.groupby('STATE')['SALES'].sum()

# get the highest sales value of the categories
men_state_sales.sort_values(ascending=False).head(1)

**Delaware** was the state with the highest men's product sales of **2.3 million**.

###Question 4: What retailer purchased the most units in 2021? In 2020?

In [ ]:
# make a variable for 2021 units purchased
retailer_units_2021 = sales_data[
    sales_data['YEAR'] == 2021
].groupby('RETAILER')['UNITS_SOLD'].sum()

# sort the values in descending order
retailer_units_2021.sort_values(ascending=False).head(1)

In [ ]:
# do the same for 2020
retailer_units_2020 = sales_data[
    sales_data['YEAR'] == 2020
].groupby('RETAILER')['UNITS_SOLD'].sum()

# sort the data in descending order
retailer_units_2020.sort_values(ascending=False).head(1)

**Foot Locker** was the retailer with the most units sold in 2021, and **Amazon** was the retailer with the most units sold in 2020.

##Additional Insights

#### Q: What were the sales by year? Did sales decrease or increase?

In [ ]:
# group the sales data by year and sum up all sales
sales_data.groupby('YEAR')['SALES'].sum()

Sales in 2020 were 24.2 million while sales in 2021 were 98.1 million. So 2021 had significantly more sales.

In [ ]:
# use group by to get sales data by year
yearly_sales = sales_data.groupby('YEAR')['SALES'].sum()

# calculate YoY growth
growth = (
    (yearly_sales[2021] - yearly_sales[2020])
    / yearly_sales[2020]
) * 100

growth

Sales increased 305% YoY from 2020 to 2021.

#### Q: How did sales change over the months?

In [ ]:
# pull monthly sales by grouping by month and year summed
monthly_sales = sales_data.groupby(
    ['YEAR', 'MONTH']
)['SALES'].sum()

monthly_sales

In [ ]:
# view for what month generated the highest sales overall, not including year
sales_data.groupby('MONTH')['SALES'].sum().sort_values(ascending=False)

July generated the highest combined sales across 2020 and 2021.

#### Q: What were the sales per retailer?

In [ ]:
# use group by to look at sales per retailer
retailer_sales = sales_data.groupby('RETAILER')['SALES'].sum()

# sort the values in descending order
retailer_sales.sort_values(ascending=False)

Foot Locker generated the highest total sales, with approximately 53.6 million in revenue.

In [ ]:
# make a group by function to sort by year and retailer
sales_data.groupby(
    ['YEAR', 'RETAILER']
)['SALES'].sum().sort_values(ascending=False)

#### Q: What are the sales per region?

In [ ]:
# create a variable for region sales
region_sales = sales_data.groupby('REGION')['SALES'].sum()

region_sales.sort_values(ascending=False)

The Northeast seemed to generate the most sales, followed by the Midwest. The South generated the least amount of sales.

#### Q: What sales method generated the most sales dollars?

In [ ]:
sales_data.groupby('SALES_METHOD')['SALES'].sum().sort_values(ascending=False)

Online generated the most followed by outlet.

#### Q: Which products or retailers generate the highest operating margins?

In [ ]:
# group by product name and operating margin mean
sales_data.groupby('PRODUCT_NAME')['OPERATING_MARGIN'].mean().sort_values(
    ascending=False
)

Men's Street Footwear had the highest operating margin at 45%.

In [ ]:
# group by retailer and operating margin
sales_data.groupby('RETAILER')['OPERATING_MARGIN'].mean().sort_values(
    ascending=False
)

Sports Direct had the highest average operating margin at 50%, closely followed by Walmart at 47.5%.